# Lloyd's Heuristic

#### Libraries

In [33]:
from importlib import import_module
from tabulate import tabulate

from src import init, metrics
from src.algorithms import lloyd

#### Parameters

In [34]:
DATASET_NAME = 'motor'           # ['adult', 'compas', 'crime', 'motor']
N_CLUSTERS = 4
RANDOM_STATE = 4
INIT_METHOD = 'kmeans_plusplus'

#### Load dataset

In [35]:
dataset_module = import_module(f"src.datasets.{DATASET_NAME}")
_, X, s, n_nonsensitive = dataset_module.load()

Loading the processed Motor Insurance dataset (motor) from 'data/datasets/motor/motor.csv'
╭───────────┬─────────────┬──────────────┬─────────────────────────────┬───────────────────────╮
│   dataset │   # objects │   # features │   # nonsensitive attributes │   sensitive attribute │
├───────────┼─────────────┼──────────────┼─────────────────────────────┼───────────────────────┤
│     motor │       36311 │          521 │                          11 │                   Age │
╰───────────┴─────────────┴──────────────┴─────────────────────────────┴───────────────────────╯


#### Load initialised centroids

In [36]:
init_centroids = init.load(
    dataset_name=DATASET_NAME, n_clusters=N_CLUSTERS,
    init_method=INIT_METHOD, random_state=RANDOM_STATE
    )

Loading centroids from 'data/init/motor/k=4/kmeans_plusplus/motor.k4.kmeans_plusplus.r4.csv'
  shape: (4, 521)


#### Run Lloyd's heuristic

In [37]:
c, centroids, info, stats = lloyd.run(
    X=X.copy(), n_nonsensitive=n_nonsensitive, n_clusters=N_CLUSTERS,
    init_centroids=init_centroids, dataset_name=DATASET_NAME,
    init_method=INIT_METHOD, random_state=RANDOM_STATE
    )

Initiating Lloyd's heuristic
╭───────────┬──────────────┬─────────────────┬────────────────┬─────────────┬────────────┬────────╮
│   dataset │   n_clusters │     init_method │   random_state │   algorithm │   max_iter │    tol │
├───────────┼──────────────┼─────────────────┼────────────────┼─────────────┼────────────┼────────┤
│     motor │            4 │ kmeans_plusplus │              4 │       Lloyd │        100 │ 0.0001 │
╰───────────┴──────────────┴─────────────────┴────────────────┴─────────────┴────────────┴────────╯
Running algorithm
╭────────┬─────────────┬─────────────────┬──────────────────┬──────────────────┬───────────────────┬──────────────────╮
│   iter │   objective │   reassignments │   centroid shift │   centroids time │   assignment time │   iteration time │
├────────┼─────────────┼─────────────────┼──────────────────┼──────────────────┼───────────────────┼──────────────────┤
│      1 │ 7.30024e-01 │           36311 │                - │                - │          228

#### Evaluate

In [38]:
import pandas as pd
with pd.ExcelWriter('./result/motor/Kmeans_K4_4.xlsx') as writer:
    c.to_excel(writer, sheet_name='Cluster', index=False)
    centroids.to_excel(writer, sheet_name='Centroid', index=False)
    info.to_excel(writer, sheet_name='Info', index=False)
    stats.to_excel(writer, sheet_name='Stats', index=False)

In [39]:
scores = metrics.evaluate(
    X=X, n_nonsensitive=n_nonsensitive, s=s, c=c, centroids=centroids,
    n_clusters=N_CLUSTERS, window_size=3
    )
print(tabulate(scores.to_frame(), tablefmt='rounded_outline'))

╭─────────────────────────────────────┬───────────╮
│ k-means objective                   │ 0.443078  │
│ max ks statistic                    │ 0.1004    │
│ max emd                             │ 0.0507362 │
│ pooling window loss (size=3)        │ 0.125427  │
│ Modified Abraham (2020)'s deviation │ 1.33377   │
│ Ziko (2021)'s fairness error        │ 0.974581  │
│ Bera (2019)'s generalised balance   │ 0.144155  │
╰─────────────────────────────────────┴───────────╯


In [40]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

# Plot the overall dataset distribution (dashed black)
sns.kdeplot(data=s, fill=False, color="black", linestyle="--", label="dataset")  # data1 as in data 2 we dont have age column. Also it will not affect as it drawing for whole dataset

# Plot each cluster's age distribution
clusters = sorted(c.unique())
palette = sns.color_palette("tab10", len(clusters))

for i, j in enumerate(clusters):
    subset = s[c == j]  # here for which points we will need original dataset or data1
    sns.kdeplot(subset, fill=False, color=palette[i], label=f"cluster_{i} ({len(subset)} objects)")

plt.xlabel("age")
plt.ylabel("proportion")
plt.title("Age Distribution : Clusters vs Full Dataset")
plt.legend()
plt.tight_layout()
#plt.show()
plt.savefig("./result/motor/Plot_Kmeans_K4_4.png")
plt.close()